# 09 — Tuning fine: LR piu' basso, piu' epoche, early stopping
`FREEZE_LAYERS=0` confermato il migliore (0.701 mean / 0.692 fusione, contro ~0.60 con freeze=8). Qui NON si tocca il freeze — si affina il training stesso:

| Parametro | Prima | Ora | Perche' |
|---|---|---|---|
| LR | 3e-5 | **2e-5** | piu' conservativo, standard per corpus piccoli |
| EPOCHS | 6 | **10** | sicuro con `load_best_model_at_end`, il rischio e' solo tempo |
| weight_decay | 0.01 | **0.05** | regolarizzazione piu' dolce del freeze |
| EarlyStopping | assente | **patience=3** | si ferma da solo se non migliora, risparmia GPU |

Include entrambi gli esperimenti (mean pooling da sola + fusione end-to-end), stessa architettura del notebook 08.

In [9]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import glob, random, re
import numpy as np, pandas as pd
import torch, torch.nn as nn
from transformers import (AutoTokenizer, AutoModel, Trainer, TrainingArguments,
                          DataCollatorWithPadding, EarlyStoppingCallback)
from transformers.modeling_outputs import SequenceClassifierOutput
from datasets import Dataset
from sklearn.utils.class_weight import compute_class_weight
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score, classification_report

In [10]:
SEED = 123
def set_all_seeds(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_all_seeds(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else
                      ("mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu"))
print(f"[device] {device}" + (f" -> {torch.cuda.get_device_name(0)}" if device.type=='cuda' else ""))

[device] cuda -> Tesla T4


In [11]:
import glob

CANDIDATE_PATHS = [
    "/kaggle/input/**/dataset_processed_quantile1_sentences.csv",   # Kaggle
    "../Dati/Processed/dataset_processed_quantile1_sentences.csv",  # locale, lanciato da Codice/
    "Dati/Processed/dataset_processed_quantile1_sentences.csv",     # locale, lanciato dalla cartella madre
    "dataset_processed_quantile1_sentences.csv",                   # locale, file nella stessa cartella del notebook
]

CSV_PATH = None
for pattern in CANDIDATE_PATHS:
    hits = glob.glob(pattern, recursive=True)
    if hits:
        CSV_PATH = hits[0]
        break

if CSV_PATH is None:
    raise FileNotFoundError(
        "dataset_processed_quantile1_sentences.csv non trovato. Su Kaggle: controlla di aver collegato "
        "il dataset come Input. In locale: lancia dalla cartella Codice/ (come gli altri script), "
        "oppure copia il CSV nella stessa cartella del notebook."
    )
print(f"[data] uso: {CSV_PATH}")
MODEL_NAME = "bert-base-cased"
FREEZE_LAYERS = 0   # confermato il migliore, non si tocca piu'

df = pd.read_csv(CSV_PATH)
KEEP_COLS = ["article_id", "topic_id", "binary_label", "fold", "text_bert", "full_text", "pos_text"]
df = df[KEEP_COLS].reset_index(drop=True)
print(f"[shape] {df.shape[0]} docs")

[shape] 978 docs


In [12]:
MAX_LEN = 512
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
base_cols = ["text_bert", "binary_label", "fold", "article_id"]
ds_full = Dataset.from_pandas(df[base_cols].reset_index(drop=True), preserve_index=False)
ds_full = ds_full.map(lambda b: tok(b["text_bert"], truncation=True, max_length=MAX_LEN), batched=True)
ds_full = ds_full.rename_column("binary_label", "labels")
MODEL_COLS = ["input_ids", "attention_mask", "token_type_ids", "labels"]

def make_fold(k):
    folds = ds_full["fold"]
    train_idx = [i for i, f in enumerate(folds) if f != k]
    eval_idx  = [i for i, f in enumerate(folds) if f == k]
    train_ds, eval_ds = ds_full.select(train_idx), ds_full.select(eval_idx)
    eval_ids, eval_labels = list(eval_ds["article_id"]), list(eval_ds["labels"])
    drop = [c for c in train_ds.column_names if c not in MODEL_COLS]
    return (train_ds.remove_columns(drop), eval_ds.remove_columns(drop), eval_ids, eval_labels)

Map:   0%|          | 0/978 [00:00<?, ? examples/s]

## WeightedTrainer + metriche + training_args (con LR/epochs/weight_decay aggiornati)

In [13]:
def fold_class_weights(train_labels):
    w = compute_class_weight("balanced", classes=np.array([0,1]), y=np.array(train_labels))
    return torch.tensor(w, dtype=torch.float)

class WeightedTrainer(Trainer):
    def __init__(self, *a, class_weights=None, **kw):
        super().__init__(*a, **kw); self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        w = self.class_weights.to(outputs.logits.device) if self.class_weights is not None else None
        loss = nn.functional.cross_entropy(outputs.logits, labels, weight=w)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"f1_macro": f1_score(labels, preds, average="macro"), "accuracy": accuracy_score(labels, preds),
            "f1_class0": f1_score(labels, preds, pos_label=0), "f1_class1": f1_score(labels, preds, pos_label=1)}

LR, EPOCHS, TRAIN_BS, EVAL_BS, N_FOLDS = 2e-5, 10, 16, 32, 5   # LR piu' basso, piu' epoche

def training_args(tag):
    return TrainingArguments(
        output_dir=f"./bert_{tag}", learning_rate=LR, num_train_epochs=EPOCHS,
        per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
        weight_decay=0.05, warmup_ratio=0.1,   # weight_decay alzato da 0.01
        eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True, metric_for_best_model="f1_macro", greater_is_better=True, save_total_limit=1,
        logging_strategy="epoch", report_to="none", seed=SEED, fp16=torch.cuda.is_available())

def early_stop():
    return [EarlyStoppingCallback(early_stopping_patience=3)]

# Esperimento 1 — Mean pooling, tuning fine (LR=2e-5, 10 epoche, early stopping)

In [14]:
class BertMeanPoolClassifier(nn.Module):
    def __init__(self, model_name, num_labels=2, freeze_layers=0):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        if freeze_layers > 0:
            for p in self.bert.embeddings.parameters(): p.requires_grad = False
            for layer in self.bert.encoder.layer[:freeze_layers]:
                for p in layer.parameters(): p.requires_grad = False
        self.dropout = nn.Dropout(0.15)   # leggermente piu' alto di prima (0.1)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None, **kw):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        return SequenceClassifierOutput(logits=self.classifier(self.dropout(pooled)))

collator = DataCollatorWithPadding(tokenizer=tok)
oof_rows = []
for k in range(N_FOLDS):
    print(f"\n===== FOLD {k} — mean pooling, tuning fine =====")
    train_ds, eval_ds, eval_ids, eval_labels = make_fold(k)
    torch.manual_seed(SEED)
    model = BertMeanPoolClassifier(MODEL_NAME, freeze_layers=FREEZE_LAYERS).to(device)
    weights = fold_class_weights(train_ds["labels"])
    trainer = WeightedTrainer(model=model, args=training_args(f"mean_tuned_fold{k}"),
                              train_dataset=train_ds, eval_dataset=eval_ds,
                              data_collator=collator, class_weights=weights, compute_metrics=compute_metrics,
                              callbacks=early_stop())
    trainer.train()
    y_pred = np.argmax(trainer.predict(eval_ds).predictions, axis=-1)
    print(f"[fold {k}] F1 macro = {f1_score(eval_labels, y_pred, average='macro'):.3f}")
    oof_rows += [{"y_true": yt, "y_pred": yp} for yt, yp in zip(eval_labels, y_pred)]

oof_df = pd.DataFrame(oof_rows)
print(f"\n=== MEAN pooling, tuning fine — F1 macro OOF: "
      f"{f1_score(oof_df.y_true, oof_df.y_pred, average='macro'):.3f} ===")
print(classification_report(oof_df.y_true, oof_df.y_pred, digits=3))


===== FOLD 0 — mean pooling, tuning fine =====


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy,F1 Class0,F1 Class1
1,0.696046,0.640493,0.665332,0.723077,0.526316,0.804348
2,0.622873,0.576853,0.615223,0.615385,0.607330,0.623116
3,0.540678,0.522339,0.724431,0.738462,0.662252,0.786611
4,0.283397,0.677960,0.697594,0.702564,0.658824,0.736364
5,0.192537,0.969757,0.654630,0.656410,0.629834,0.679426
6,0.082478,1.166234,0.724763,0.743590,0.652778,0.796748
7,0.021005,1.421209,0.756554,0.789744,0.666667,0.846442
8,0.016273,1.532051,0.727775,0.743590,0.662162,0.793388
9,0.000474,1.597171,0.738799,0.748718,0.687898,0.789700
10,0.000359,1.591391,0.738799,0.748718,0.687898,0.789700


[fold 0] F1 macro = 0.757

===== FOLD 1 — mean pooling, tuning fine =====


...

[fold 3] F1 macro = 0.698

===== FOLD 4 — mean pooling, tuning fine =====


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy,F1 Class0,F1 Class1
1,0.693311,0.674865,0.462494,0.615385,0.175824,0.749164
2,0.609074,0.609501,0.604754,0.605128,0.592593,0.616915
3,0.462250,0.608195,0.723321,0.758974,0.624000,0.822642
4,0.259530,0.668378,0.702254,0.738462,0.598425,0.806084
5,0.115299,1.093748,0.663123,0.666667,0.628571,0.697674
6,0.071330,1.417073,0.731730,0.743590,0.675325,0.788136
7,0.016506,1.738660,0.702255,0.733333,0.606061,0.798450
8,0.003164,1.738492,0.696750,0.702564,0.654762,0.738739
9,0.000320,1.691959,0.712735,0.723077,0.658228,0.767241


[fold 4] F1 macro = 0.732

=== MEAN pooling, tuning fine — F1 macro OOF: 0.737 ===
              precision    recall  f1-score   support

           0      0.618     0.715     0.663       326
           1      0.845     0.779     0.811       652

    accuracy                          0.758       978
   macro avg      0.732     0.747     0.737       978
weighted avg      0.770     0.758     0.762       978



# Esperimento 2 — Fusione end-to-end, tuning fine

In [15]:
def psycholinguistic_features(text):
    words = text.split(); n = max(len(words), 1)
    return [100*text.count("!")/n, 100*text.count("?")/n,
            100*len(re.findall(r"\b(not|never|no|n't)\b", text, re.I))/n,
            100*len(re.findall(r"\b(i|we|my|our|us)\b", text, re.I))/n,
            100*len(re.findall(r"\b(you|your)\b", text, re.I))/n,
            100*len(re.findall(r"\b(could|would|should|must|might)\b", text, re.I))/n,
            np.mean([len(w) for w in words]) if words else 0,
            len(set(w.lower() for w in words))/n]

psy = np.array([psycholinguistic_features(t) for t in df["full_text"]])
FEAT_DIM = psy.shape[1] + 50

def build_fold_features(train_idx):
    pos_vec = TfidfVectorizer(ngram_range=(2, 2), max_features=50, min_df=3)
    pos_vec.fit(df["pos_text"].iloc[train_idx])
    pos_all = pos_vec.transform(df["pos_text"]).toarray()
    sc = StandardScaler().fit(np.hstack([psy[train_idx], pos_all[train_idx]]))
    return sc.transform(np.hstack([psy, pos_all]))

MODEL_COLS_FUS = MODEL_COLS + ["features"]

def make_fold_fusion(k, feat_matrix):
    folds = ds_full["fold"]
    train_idx = [i for i, f in enumerate(folds) if f != k]
    eval_idx  = [i for i, f in enumerate(folds) if f == k]
    train_ds = ds_full.select(train_idx).add_column("features", feat_matrix[train_idx].tolist())
    eval_ds  = ds_full.select(eval_idx).add_column("features", feat_matrix[eval_idx].tolist())
    eval_ids, eval_labels = list(eval_ds["article_id"]), list(eval_ds["labels"])
    drop = [c for c in train_ds.column_names if c not in MODEL_COLS_FUS]
    return (train_ds.remove_columns(drop), eval_ds.remove_columns(drop), eval_ids, eval_labels)

class FusionCollator:
    def __init__(self, tokenizer):
        self.base = DataCollatorWithPadding(tokenizer=tokenizer)
    def __call__(self, batch):
        feats = torch.tensor([b.pop("features") for b in batch], dtype=torch.float)
        out = self.base(batch)
        out["features"] = feats
        return out

class BertFusionClassifier(nn.Module):
    def __init__(self, model_name, feat_dim, num_labels=2, freeze_layers=0):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        if freeze_layers > 0:
            for p in self.bert.embeddings.parameters(): p.requires_grad = False
            for layer in self.bert.encoder.layer[:freeze_layers]:
                for p in layer.parameters(): p.requires_grad = False
        hidden = self.bert.config.hidden_size
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Sequential(
            nn.Linear(hidden + feat_dim, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, num_labels))
    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, features=None, labels=None, **kw):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        combined = torch.cat([self.dropout(pooled), features], dim=1)
        return SequenceClassifierOutput(logits=self.classifier(combined))

In [16]:
fusion_collator = FusionCollator(tok)
oof_rows_fus = []
for k in range(N_FOLDS):
    print(f"\n===== FOLD {k} — fusione end-to-end, tuning fine =====")
    train_idx_k = [i for i, f in enumerate(df["fold"]) if f != k]
    feat_matrix_k = build_fold_features(train_idx_k)
    train_ds, eval_ds, eval_ids, eval_labels = make_fold_fusion(k, feat_matrix_k)

    torch.manual_seed(SEED)
    model = BertFusionClassifier(MODEL_NAME, FEAT_DIM, freeze_layers=FREEZE_LAYERS).to(device)
    weights = fold_class_weights(train_ds["labels"])
    trainer = WeightedTrainer(model=model, args=training_args(f"fusion_tuned_fold{k}"),
                              train_dataset=train_ds, eval_dataset=eval_ds,
                              data_collator=fusion_collator, class_weights=weights, compute_metrics=compute_metrics,
                              callbacks=early_stop())
    trainer.train()
    y_pred = np.argmax(trainer.predict(eval_ds).predictions, axis=-1)
    print(f"[fold {k}] F1 macro = {f1_score(eval_labels, y_pred, average='macro'):.3f}")
    oof_rows_fus += [{"y_true": yt, "y_pred": yp} for yt, yp in zip(eval_labels, y_pred)]

oof_df_fus = pd.DataFrame(oof_rows_fus)
print(f"\n=== Fusione END-TO-END, tuning fine — F1 macro OOF: "
      f"{f1_score(oof_df_fus.y_true, oof_df_fus.y_pred, average='macro'):.3f} ===")
print(classification_report(oof_df_fus.y_true, oof_df_fus.y_pred, digits=3))


===== FOLD 0 — fusione end-to-end, tuning fine =====


Flattening the indices:   0%|          | 0/783 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/195 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Flattening the indices:   0%|          | 0/195 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy,F1 Class0,F1 Class1
1,0.688908,0.688826,0.493379,0.682051,0.184211,0.802548
2,0.650901,0.676979,0.405765,0.430769,0.527660,0.283871
3,0.600522,0.590078,0.676581,0.692308,0.605263,0.747899
4,0.415418,0.619014,0.668071,0.697436,0.569343,0.766798
5,0.295880,0.881125,0.605128,0.605128,0.605128,0.605128
6,0.195884,0.856282,0.670122,0.676923,0.622754,0.717489


[fold 4] F1 macro = 0.677

=== Fusione END-TO-END, tuning fine — F1 macro OOF: 0.721 ===
              precision    recall  f1-score   support

           0      0.575     0.776     0.661       326
           1      0.864     0.713     0.782       652

    accuracy                          0.734       978
   macro avg      0.720     0.745     0.721       978
weighted avg      0.768     0.734     0.741       978



## Tabella finale aggiornata

| Variante | Dataset | F1 macro OOF |
|---|---|---|
| Shallow TF-IDF | 208 topic (originale) | 0.66 |
| BERT frozen | 208 topic (originale) | 0.61 |
| CLS pooling (Luca) | 208 topic (originale) | 0.694 |
| Fusione post-hoc | 208 topic (originale) | 0.603 |
| Fusione e2e, freeze=8 | 208 topic (originale) | 0.602 |
| Fusione e2e, freeze=0, LR=3e-5/6ep | 208 topic (originale) | 0.692 |
| Mean pooling, freeze=8 | 208 topic (originale) | 0.608 |
| Mean pooling, freeze=0, LR=3e-5/6ep | 208 topic (originale) | 0.701 |
| **Mean pooling, tuning fine (qui)** | **326 topic (espanso)** | **0.737** |
| **Fusione e2e, tuning fine (qui)** | **326 topic (espanso)** | **0.721** |

**Nota metodologica**: le prime 8 righe sono state stabilite sul dataset originale (208 topic / 624 righe),
prima dell'espansione dello scraping. Le ultime due righe (questo notebook) girano sul dataset espanso
(326 topic / 978 righe, dopo il secondo giro di scraping e il ri-preprocessing). Il confronto diretto tra le
due sezioni non è quindi a parità di dati — l'aumento da 0.701 a 0.737 include sia l'effetto degli
iperparametri ottimizzati (LR più basso, più epoche, early stopping) sia l'effetto dei dati aggiuntivi, non
scomponibile a posteriori senza un run di controllo sullo stesso dataset piccolo.

**Risultato finale**: F1 macro OOF = 0.737 (mean pooling), confermato non overfittato tramite verifica
separata su holdout test set (notebook 10: F1 test = 0.685, praticamente identico alla validation = 0.684).
